*Notebook Last Validated: 2026-10-27*

Project UID (Internal, Prod):ae645d92-06ad-420b-9d0a-c4209fd0f073

# Tutorial 2 Notebook

This notebook corresponds to Step 4 of Tutorial 2.

## 1. Load All Necessary Libraries

Ensure that you are running this notebook in the correct kernel.
If needed, install the required libraries by uncommenting the following line. 

In [ ]:
# !pip install --upgrade rhino_health

In [ ]:
import pandas as pd
from datetime import datetime
import os
import sys
from pathlib import Path
from getpass import getpass
import rhino_health as rh
from rhino_health.lib.endpoints.dataset.dataset_dataclass import DatasetCreateInput
from rhino_health.lib.endpoints.code_object.code_object_dataclass import (
    CodeObjectCreateInput,
    CodeTypes,
    CodeObjectRunInput,
)
from rhino_health import ApiEnvironment

print('Done')

## 2. Log in to the Rhino FCP


Replace the value of the `my_username` variable with your Rhino FCP username.

Replace the value of `project_uid` with the UID of the project you just created. You can find this by clicking on the three vertical dot button in your project's square, and then selecting the botton that says `Copy UID`

Upon running the cell, you will be prompted to provide your password.

In [ ]:
my_username = '<ENTER_YOUR_EMAIL>' # REPLACE                                          
project_uid = "<ENTER_YOUR_PROJECT_UID>" # REPLACE

print("Logging In")
session = rh.login(username=my_username, password=getpass())
print("Logged In")

## 3. Load the Datasets

### 3.1  Collect Necessary Parameters

In [ ]:
workgroup = session.project.get_collaborating_workgroups(project_uid)[0]
dataschema = session.project.get_data_schemas(project_uid)[0]
print(f"Loaded dataschema '{dataschema.name}' with uid '{dataschema.uid}'")

# If you followed Step 2.2A (Client-Mounted Storage), uncomment & run this
# Replace the paths here according to your client-mounted storage paths
dataset1_part1_path = "/rhino_data/external/import-external-datasets-dev/user-resource-examples/tutorial2/site1_part1_dataset.csv"
dataset2_part1_path = "/rhino_data/external/import-external-datasets-dev/user-resource-examples/tutorial2/site2_part1_dataset.csv"
dataset3_part1_path = "/rhino_data/external/import-external-datasets-dev/user-resource-examples/tutorial2/site3_part1_dataset.csv"

# Alternatively, if you followed Step 2.2B (SFTP), uncomment & run this
# Replace the paths here according to where you placed the files on your client
# dataset1_part1_path = "/rhino_data/tutorial_2/datasets/site1_part1_dataset.csv"
# dataset2_part1_path = "/rhino_data/tutorial_2/datasets/site2_part1_dataset.csv"
# dataset3_part1_path = "/rhino_data/tutorial_2/datasets/site3_part1_dataset.csv"

print("Loaded: Dataset Part 1 Paths")

### 3.2 Trigger Dataset Registration (Part 1)

In [ ]:
dataset_creation_params = DatasetCreateInput(
    name="Site 1 Dataset",
    description="Diabetes dataset for site 1",
    project_uid=project_uid, 
    workgroup_uid=workgroup.uid,
    data_schema_uid=dataschema.uid,
    csv_filesystem_location=dataset1_part1_path,
    is_data_deidentified=True,
    method="filesystem",
)

site1_dataset = session.dataset.add_dataset(dataset_creation_params)
print(f"Created new dataset '{site1_dataset.name}' with uid '{site1_dataset.uid}'")

dataset_creation_params = DatasetCreateInput(
    name="Site 2 Dataset",
    description="Diabetes dataset for site 2",
    project_uid=project_uid, 
    workgroup_uid=workgroup.uid,
    data_schema_uid=dataschema.uid,
    csv_filesystem_location=dataset2_part1_path,
    is_data_deidentified=True,
    method="filesystem",
)

site2_dataset = session.dataset.add_dataset(dataset_creation_params)
print(f"Created new dataset '{site2_dataset.name}' with uid '{site2_dataset.uid}'")

dataset_creation_params = DatasetCreateInput(
    name="Site 3 Dataset",
    description="Diabetes dataset for site 3",
    project_uid=project_uid, 
    workgroup_uid=workgroup.uid,
    data_schema_uid=dataschema.uid,
    csv_filesystem_location=dataset3_part1_path,
    is_data_deidentified=True,
    method="filesystem",
)

site3_dataset = session.dataset.add_dataset(dataset_creation_params)
print(f"Created new dataset '{site3_dataset.name}' with uid '{site3_dataset.uid}'")
print("You should now have 3 new datasets in the project within the GUI. Feel free to take a look!")

## 4. Harmonize Datasets Using Containerless GC

By reviewing the dataset analytics on the Dashboard UI, you can see several inconsistencies across the 3 datasets.
1. Outcome - we see a mix of values (0, 1, Negative, Positive)
2. Weight - it looks like Dataset 1 is on a different unit scale
3. Height - it looks like Dataset 2 is on a different unit scale
4. SkinThickness - it looks like Dataset 2 is on a different unit scale

These kinds of inconsistencies can often occur when collecting data from multiple sources. 

![Part 1 Data Discrepancies](part_1_data_discrepancies.png)

In this next part, you will use simple pandas operations to produce harmonized versions of these datasets. 

### 4.1 Define Harmonization Code for Each Dataset

In [ ]:
site_1_code = "df.replace({'Outcome': { 'Positive': 1, 'Negative': 0}}, inplace=True)\ndf.Weight = round(df.Weight*0.453592, 0).astype(int)"
site_2_code = "df['SkinThickness'] = df['SkinThickness']*100\ndf['Height'] = df['Height']/100"
site_3_code = "df.replace({'Outcome': { 'Positive': 1, 'Negative': 0}},inplace=True)\ndf['Pregnancies'].replace('None', 0, inplace=True)"
print("Code Loaded")

### 4.2 Site 1 Data Harmonization

In [ ]:
print("Starting to run harmonization on Site 1 data")
output_dataset, run_results = site1_dataset.run_code(site_1_code, output_data_schema_uid=dataschema.uid, output_dataset_names_suffix=" Fixed")
print("Finished running harmonization on Site 1 data")

print("You can now see a new dataset in the Dashboard UI named 'Site 1 Dataset Fixed'")
print("View the Results below:")
run_results.dict()

### 4.3 Site 2 Data Harmonization

In [ ]:
print("Starting to run harmonization on Site 2 data")
output_dataset, run_results = site2_dataset.run_code(site_2_code, output_data_schema_uid=dataschema.uid, output_dataset_names_suffix=" Fixed")
print("Finished running harmonization on Site 2 data")

print("You can now see a new dataset in the Dashboard UI named 'Site 2 Dataset Fixed'")
print("View the Results below:")
run_results.dict()

### 4.4 Site 3 Data Harmonization

In [ ]:
print("Starting to run harmonization on Site 3 data")
output_dataset, run_results = site3_dataset.run_code(site_3_code, 
                                    output_data_schema_uid=dataschema.uid, 
                                    output_dataset_names_suffix=" Fixed")
print("Finished running harmonization on Site 3 data")

print("You can now see a new dataset in the Dashboard UI named 'Site 3 Dataset Fixed'")
print("View the Results below:")
run_results.dict()

### 4.5 Review Harmonized Outputs

Return to Dataset Analytics in the Dashboard UI and select the three “Fixed” output Datasets to view the harmonized results.

![Part 1 Data Fixed](part_1_data_fixed.png)

## 5. Register Updated Datasets (Part 2)

Now let's imagine you have some updated data (`*_part_2.csv` files). That you would like to harmonize in a similar fashion. Making simple modifications to the code above we can harmonize the new data with little effort.

First let's import the updated data as new datasets.

In [ ]:
# If you followed Step 2.2A (Client-Mounted Storage), uncomment & run this
# Replace the paths here according to your client-mounted storage paths
dataset1_part2_path = "/rhino_data/external/import-external-datasets-dev/user-resource-examples/tutorial2/site1_part2_dataset.csv"
dataset2_part2_path = "/rhino_data/external/import-external-datasets-dev/user-resource-examples/tutorial2/site2_part2_dataset.csv"
dataset3_part2_path = "/rhino_data/external/import-external-datasets-dev/user-resource-examples/tutorial2/site3_part2_dataset.csv"

# Alternatively, if you followed Step 2.2B (SFTP), uncomment & run this
# Replace the paths here according to where you placed the files on your client
# dataset1_part2_path = "/rhino_data/tutorial_2/datasets/site1_part2_dataset.csv"
# dataset2_part2_path = "/rhino_data/tutorial_2/datasets/site2_part2_dataset.csv"
# dataset3_part2_path = "/rhino_data/tutorial_2/datasets/site3_part2_dataset.csv"

print("Loaded: Dataset Part 2 Paths")

In [ ]:
dataset_creation_params = DatasetCreateInput(
    name="Site 1 Dataset - Part 2",
    description="Updated diabetes dataset for site 1 - part 2",
    project_uid=project_uid, 
    workgroup_uid=workgroup.uid,
    data_schema_uid=dataschema.uid,
    csv_filesystem_location=dataset1_part2_path,
    is_data_deidentified=True,
    method="filesystem",
)
site1_part2_dataset = session.dataset.add_dataset(dataset_creation_params)
print(f"Created new dataset '{site1_part2_dataset.name}' with uid '{site1_part2_dataset.uid}'")

dataset_creation_params = DatasetCreateInput(
    name="Site 2 Dataset - Part 2",
    description="Updated diabetes dataset for site 2 - part 2",
    project_uid=project_uid, 
    workgroup_uid=workgroup.uid,
    data_schema_uid=dataschema.uid,
    csv_filesystem_location=dataset2_part2_path,
    is_data_deidentified=True,
    method="filesystem",
)
site2_part2_dataset = session.dataset.add_dataset(dataset_creation_params)
print(f"Created new dataset '{site2_part2_dataset.name}' with uid '{site2_part2_dataset.uid}'")

dataset_creation_params = DatasetCreateInput(
    name="Site 3 Dataset - Part 2",
    description="Updated diabetes dataset for site 3 - part 2",
    project_uid=project_uid, 
    workgroup_uid=workgroup.uid,
    data_schema_uid=dataschema.uid,
    csv_filesystem_location=dataset3_part2_path,
    is_data_deidentified=True,
    method="filesystem",
)
site3_part2_dataset = session.dataset.add_dataset(dataset_creation_params)
print(f"Created new dataset '{site3_part2_dataset.name}' with uid '{site3_part2_dataset.uid}'")

print("Done")

## 6. Harmonize Data Reusing Previous Harmonization Code

The new Part 2 datasets suffer from the same inconsistencies as the Part 1 datasets.

You can easily fix this by running the same preprocessing code you have defined earlier:

In [ ]:
# Site 1
print("Starting to run harmonization on Site 1 - Part 2 data")
output_dataset, run_results = site1_part2_dataset.run_code(site_1_code, output_data_schema_uid=dataschema.uid, output_dataset_names_suffix=" Fixed")
print("Finished running harmonization on Site 1 - Part 2 data")
print("You can now see a new dataset in the Dashboard UI named 'Site 1 Dataset - Part 2 Fixed'")

# Site 2
print("Starting to run harmonization on Site 2 - Part 2 data")
output_dataset, run_results = site2_part2_dataset.run_code(site_2_code, output_data_schema_uid=dataschema.uid, output_dataset_names_suffix=" Fixed")
print("Finished running harmonization on Site 2 - Part 2 data")
print("You can now see a new dataset in the Dashboard UI named 'Site 2 Dataset - Part 2 Fixed'")

# Site 3
print("Starting to run harmonization on Site 3 - Part 2 data")
output_dataset, run_results = site3_part2_dataset.run_code(site_3_code, output_data_schema_uid=dataschema.uid, output_dataset_names_suffix=" Fixed")
print("Finished running harmonization on Site 3 - Part 2 data")
print("You can now see a new dataset in the Dashboard UI named 'Site 3 Dataset - Part 2 Fixed'")

print("Hamonization Complete.")


### Congratulations - Your datasets are now harmonized!

Use the filters on the Dataset > Analytics tab in the Dashboard UI to visualize all the harmonized results together.

![All Data Fixed](all_data_fixed.png)

# End of Tutorial 2 Notebook. 